In [ ]:
# Unfreeze more layers for fine-tuning
for param in model.layer4.parameters():
    param.requires_grad = True

# Use differential learning rates
optimizer = optim.SGD([
    {'params': model.layer4.parameters(), 'lr': 0.001},
    {'params': model.fc.parameters(), 'lr': 0.01}
], momentum=0.9)

print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 5. Fine-tuning Strategy (Progressive Unfreezing)

In [ ]:
def train_transfer_learning(model, train_loader, test_loader, epochs=10, lr=0.001):
    from torch.optim.lr_scheduler import StepLR
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    scheduler = StepLR(optimizer, step_size=5, gamma=0.1)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    train_losses = []
    test_accuracies = []
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        train_losses.append(train_loss / len(train_loader))
        scheduler.step()
        
        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted.cpu() == labels).sum().item()
        
        accuracy = 100 * correct / total
        test_accuracies.append(accuracy)
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {train_losses[-1]:.4f}, Accuracy: {accuracy:.2f}%")
    
    return train_losses, test_accuracies

train_losses, test_accuracies = train_transfer_learning(model, train_loader, test_loader, epochs=5)

# Plot results
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(test_accuracies)
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Test Accuracy')
plt.grid(True)
plt.tight_layout()
plt.show()

## 4. Train with Learning Rate Scheduling

In [ ]:
# Freeze all layers except the final FC layer
for param in model.parameters():
    param.requires_grad = False

# Unfreeze final layer
for param in model.fc.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {sum(p.numel() for p in model.parameters()) - trainable_params:,}")

## 3. Feature Extraction Strategy (Freeze Backbone)

In [ ]:
# Load dataset
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

train_dataset = CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

# Load pre-trained model
model_name = 'resnet50'
model = getattr(models, model_name)(pretrained=True)

# Modify final layer for 10 classes
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)
model = model.to(device)

print(f"Loaded {model_name} pretrained model")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Load Dataset and Pre-trained Model

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR
from torch.utils.data import DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Available pre-trained models
available_models = ['resnet18', 'resnet50', 'vgg16', 'mobilenet_v2', 'efficientnet_b0']
print(f"Available models: {available_models}")

## 1. Import Libraries and Load Pre-trained Models

# Transfer Learning and Fine-tuning
## Leveraging Pre-trained Models for Custom Tasks

Topics covered:
- Loading pre-trained models (ResNet, VGG, EfficientNet)
- Feature extraction vs. fine-tuning strategies
- Freezing and unfreezing layers
- Learning rate scheduling
- Domain adaptation techniques
- Comparison of different transfer learning approaches